## Bronze Layer - Public Holidays API Ingestion

This notebook fetches public holidays data from an external API and stores it in the bronze layer.

**Data Source:** Nager.Date API (free, no API key required, no rate limits)

**Business Use Case:**
- Analyze holiday impact on sales patterns
- Identify sales spikes before major holidays (Christmas, Thanksgiving)
- Understand lower sales periods during national holidays
- Optimize inventory and marketing campaigns around holiday seasons

**Join Strategy (for Silver/Gold layers):**
- Join with `bronze.crm_sales_details` (sales) on date and country
- Flag orders placed on or near holidays
- Analyze pre-holiday vs post-holiday sales behavior

In [0]:
import requests
import json
from datetime import datetime, timedelta
from pyspark.sql.functions import col, lit, current_timestamp

In [0]:
# Country mapping for holidays API
# Note: Nager.Date API uses ISO country codes (GB for UK, not UK)

country_mapping = {
    "US": {"name": "United States", "iso_code": "US"},
    "GB": {"name": "United Kingdom", "iso_code": "GB"},  # UK uses GB code
    "DE": {"name": "Germany", "iso_code": "DE"},
    "FR": {"name": "France", "iso_code": "FR"},
    "CA": {"name": "Canada", "iso_code": "CA"},
    "AU": {"name": "Australia", "iso_code": "AU"}  # Added Australia
}

# Years to fetch (matching sales data range 2010-12-29 to 2014-01-28)
years = [2010, 2011, 2012, 2013, 2014]

print(f"Fetching public holidays for {len(country_mapping)} countries")
print(f"Years: {years[0]} to {years[-1]}")
print(f"Countries covered: {', '.join([c['name'] for c in country_mapping.values()])}")
print(f"\nTotal API calls: {len(country_mapping) * len(years)}")

In [0]:
# Fetch public holidays data for each country and year
import time

holidays_data = []

for country_code, country_info in country_mapping.items():
    print(f"\n🌍 {country_info['name']} ({country_code})")
    
    for year in years:
        print(f"  {year}...", end=" ")
        
        # Nager.Date Public Holidays API
        url = f"https://date.nager.at/api/v3/PublicHolidays/{year}/{country_info['iso_code']}"
        
        try:
            response = requests.get(url, timeout=10)
            response.raise_for_status()
            
            holidays = response.json()
            
            # Add metadata to each holiday
            for holiday in holidays:
                holiday["country_code"] = country_code
                holiday["country_name"] = country_info["name"]
                holiday["year"] = year
                holiday["ingestion_timestamp"] = datetime.now().isoformat()
                holidays_data.append(holiday)
            
            print(f"✓ {len(holidays)} holidays")
            
            # Small delay to be polite to the API
            time.sleep(0.5)
            
        except requests.exceptions.HTTPError as e:
            print(f"✗ HTTP Error: {e}")
        except requests.exceptions.Timeout:
            print(f"✗ Timeout")
        except Exception as e:
            print(f"✗ Error: {str(e)}")

print(f"\n{'='*60}")
print(f"✓ Successfully collected {len(holidays_data)} holidays")
print(f"✓ Covering {len(country_mapping)} countries over {len(years)} years")

In [0]:
# Convert to DataFrame and save to Bronze layer
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, DateType, BooleanType

schema = StructType([
    StructField("country_code", StringType(), True),
    StructField("country_name", StringType(), True),
    StructField("year", StringType(), True),
    StructField("date", DateType(), True),
    StructField("holiday_name", StringType(), True),
    StructField("local_name", StringType(), True),
    StructField("is_global", BooleanType(), True),
    StructField("raw_json", StringType(), True),
    StructField("ingestion_timestamp", TimestampType(), True)
])

# Prepare data for bronze table
bronze_records = [
    (
        holiday["country_code"],
        holiday["country_name"],
        str(holiday["year"]),
        datetime.strptime(holiday["date"], "%Y-%m-%d").date(),
        holiday.get("name", ""),
        holiday.get("localName", ""),
        holiday.get("global", False),
        json.dumps(holiday),  # Store full JSON for reference
        datetime.fromisoformat(holiday["ingestion_timestamp"])
    )
    for holiday in holidays_data
]

df_holidays_bronze = spark.createDataFrame(bronze_records, schema)

# Write to bronze layer
df_holidays_bronze.write.mode("overwrite").saveAsTable("bronze.holidays_api_raw")

print(f"✓ Saved {df_holidays_bronze.count()} holiday records to bronze.holidays_api_raw")
print(f"\nSample holidays per country:")
df_holidays_bronze.groupBy("country_name", "year").count().orderBy("country_name", "year").show()

In [0]:
%sql
-- Sample of holidays data
SELECT 
  date,
  country_name,
  holiday_name,
  local_name,
  is_global
FROM bronze.holidays_api_raw
WHERE year IN ('2010', '2011')
ORDER BY date, country_name
LIMIT 20